# Kural — Tamil SER: frozen Whisper encoder + attention pooling + classifier

Encoder is **frozen** (feature extractor). A learnable **attention-pooling** layer attends
to the emotional frames and ignores the 30s padding (the thing that broke mean-pooling),
then a linear **classifier**. Only attention + classifier train. EmoTa, **speaker-independent**
test, a few classes.

GPU runtime (Kaggle: T4 + Internet ON), token via Kaggle/Colab Secret `HF_TOKEN`.
EmoTa must be added as a (private) Dataset input.

In [ ]:
!pip -q install --upgrade "transformers>=4.44" "datasets[audio]>=2.20" "accelerate>=0.33" scikit-learn librosa soundfile huggingface_hub

In [ ]:
# ===== CONFIG =====
BACKBONE     = "openai/whisper-small"
HF_USERNAME  = "Venky0411"
HUB_REPO     = f"{HF_USERNAME}/whisper-small-ta-emotion"

# pick a few classes (drop 'fear' — minority, scored 0). Add it back for all 5.
LABELS       = ["angry", "happy", "neutral", "sad"]
label2id     = {l: i for i, l in enumerate(LABELS)}
id2label     = {i: l for l, i in label2id.items()}

N_TEST_SPEAKERS  = 4      # held-out speakers for the speaker-independent test
USE_ENGLISH_AUG  = False  # add RAVDESS/CREMA-D/TESS for volume (domain shift)
N_HEADS          = 8      # attention-pooling heads
EPOCHS           = 25
BATCH_SIZE       = 16
LR               = 1e-3

In [ ]:
from huggingface_hub import login
import os
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if not HF_TOKEN:
    try:  # Kaggle: Add-ons -> Secrets -> HF_TOKEN (toggle ON for this notebook)
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN); print('HF login OK — will push to the Hub')
else:
    print('No HF_TOKEN found -> training runs but will NOT push. '
          'Add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) and enable it to push.')


In [ ]:
# ===== EmoTa loader (filename: SPEAKER_UTTERANCE_emotion.wav) =====
import os, glob
from datasets import Dataset, Audio, load_dataset, concatenate_datasets

EMOTA_CODE = {"ang": "angry", "fea": "fear", "hap": "happy", "neu": "neutral", "sad": "sad"}

def find_emota():
    import zipfile
    roots = ["/kaggle/input", "/content", "/kaggle/working", "."]
    def scan(base):
        hits = glob.glob(os.path.join(base, "**", "*.wav"), recursive=True)
        return [h for h in hits
                if len(os.path.basename(h)[:-4].split("_")) >= 3
                and os.path.basename(h)[:-4].split("_")[2].lower() in EMOTA_CODE]
    for r in roots:
        h = scan(r)
        if h:
            print("found EmoTa in:", os.path.dirname(h[0])); return os.path.dirname(h[0])
    for r in roots:
        for z in glob.glob(os.path.join(r, "**", "*.zip"), recursive=True):
            try:
                with zipfile.ZipFile(z) as zf:
                    if any(nm.lower().endswith(".wav") for nm in zf.namelist()):
                        out = "/kaggle/working/_emota"; os.makedirs(out, exist_ok=True)
                        zf.extractall(out); h = scan(out)
                        if h:
                            print("extracted EmoTa to:", os.path.dirname(h[0])); return os.path.dirname(h[0])
            except Exception:
                pass
    raise FileNotFoundError("EmoTa not found under /kaggle/input — did you '+ Add Input' the dataset?")

def load_emota(emota_dir=None):
    emota_dir = emota_dir or find_emota()
    files = sorted(glob.glob(os.path.join(emota_dir, "*.wav")))
    rows = {"audio": [], "emo": [], "speaker": [], "source": []}
    for f in files:
        parts = os.path.basename(f)[:-4].split("_")
        if len(parts) < 3:
            continue
        emo = EMOTA_CODE.get(parts[2].lower())
        if emo is None:
            continue
        rows["audio"].append(f); rows["emo"].append(emo)
        rows["speaker"].append(parts[0]); rows["source"].append("emota")
    ds = Dataset.from_dict(rows).cast_column("audio", Audio(sampling_rate=16000))
    print(f"EmoTa: {len(ds)} clips, {len(set(rows['speaker']))} speakers, dir={emota_dir}")
    return ds

In [ ]:
# ===== optional English augmentation =====
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
EMO_MAP = {"neutral": "neutral", "calm": "neutral", "happy": "happy", "happiness": "happy",
           "sad": "sad", "sadness": "sad", "angry": "angry", "anger": "angry",
           "fear": "fear", "fearful": "fear", "disgust": None, "surprise": None,
           "surprised": None, "ps": None, "pleasant surprise": None}
def canon(v):
    return None if v is None else EMO_MAP.get(str(v).strip().lower().replace("_", " "))

def load_norm(name, source, config=None, revision=None, split="train"):
    ds = load_dataset(name, config, split=split, revision=revision, verification_mode="no_checks")
    cands = [c for c in ["label", "labels", "emotion", "emotions", "class"] if c in ds.column_names]
    label_col = next((c for c in cands if ds.features[c].__class__.__name__ == "ClassLabel"),
                     cands[0] if cands else None)
    feat = ds.features[label_col] if label_col else None
    is_cl = feat.__class__.__name__ == "ClassLabel" if feat is not None else False
    def _m(b):
        v = b[label_col] if label_col else None
        if is_cl and isinstance(v, int):
            v = feat.int2str(v)
        b["emo"] = canon(v); b["source"] = source
        return b
    ds = ds.map(_m).filter(lambda b: b["emo"] is not None)
    return ds.cast_column("audio", Audio(sampling_rate=16000)).select_columns(["audio", "emo", "source"])

ENGLISH = [("confit/cremad-parquet", "cremad", None, None),
           ("confit/ravdess-parquet", "ravdess", "fold1", None),
           ("AbstractTTS/TESS", "tess", None, None)]

In [ ]:
# ===== load EmoTa (required) + optional English =====
emota = load_emota()

english = None
if USE_ENGLISH_AUG:
    parts = []
    for name, src, cfg, rev in ENGLISH:
        try:
            d = load_norm(name, src, config=cfg, revision=rev); parts.append(d)
            print(f"  OK  {src:8s} {len(d):6d} rows")
        except Exception as e:
            print(f"  SKIP {src}: {type(e).__name__}: {str(e)[:80]}")
    english = concatenate_datasets(parts) if parts else None

In [ ]:
# ===== keep selected classes + speaker-independent split =====
from collections import Counter
keep = set(LABELS)
emota = emota.filter(lambda b: b["emo"] in keep)

spks = sorted(set(emota["speaker"]))
test_spks = set(spks[-N_TEST_SPEAKERS:])
print("held-out test speakers:", sorted(test_spks))
emota_test  = emota.filter(lambda b: b["speaker"] in test_spks).remove_columns("speaker")
emota_train = emota.filter(lambda b: b["speaker"] not in test_spks).remove_columns("speaker")

train_parts = ([english] if (USE_ENGLISH_AUG and english is not None) else []) + [emota_train]
train_ds = concatenate_datasets(train_parts).filter(lambda b: b["emo"] in keep).shuffle(seed=42)
tamil_test = emota_test
print("train:", len(train_ds), "| tamil_test:", len(tamil_test))
print("train labels:", dict(Counter(train_ds["emo"])))
print("test  labels:", dict(Counter(tamil_test["emo"])))

In [ ]:
# ===== feature extraction: log-mel + attention mask (to ignore padding) =====
from transformers import WhisperFeatureExtractor
fe = WhisperFeatureExtractor.from_pretrained(BACKBONE)

def prep(b):
    out = fe(b["audio"]["array"], sampling_rate=16000, return_attention_mask=True)
    b["input_features"] = out.input_features[0]
    b["attention_mask"] = out.attention_mask[0]
    b["labels"] = label2id[b["emo"]]
    return b

train_ds = train_ds.map(prep, remove_columns=train_ds.column_names, num_proc=2)
tamil_test = tamil_test.map(prep, remove_columns=tamil_test.column_names, num_proc=2)

In [ ]:
# ===== model: frozen Whisper encoder -> attention pooling -> classifier =====
import torch, torch.nn as nn
from transformers import WhisperModel

class WhisperAttnSER(nn.Module):
    def __init__(self, backbone, num_labels, n_heads=8, dropout=0.1):
        super().__init__()
        self.encoder = WhisperModel.from_pretrained(backbone).encoder
        for p in self.encoder.parameters():
            p.requires_grad = False                       # FROZEN encoder
        d = self.encoder.config.d_model
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d, num_labels)

    def forward(self, input_features, attention_mask=None, labels=None):
        self.encoder.eval()
        with torch.no_grad():                             # encoder is not trained
            enc = self.encoder(input_features).last_hidden_state   # (B, T, d)
        kpm = None
        if attention_mask is not None:
            m = attention_mask[:, ::2][:, :enc.size(1)]    # mel 3000 -> encoder 1500
            kpm = (m == 0)                                 # True = padding -> ignored
            kpm[kpm.all(dim=1)] = False                    # guard fully-masked rows
        q = self.query.expand(enc.size(0), -1, -1)
        pooled, _ = self.attn(q, enc, enc, key_padding_mask=kpm)   # attend to salient frames
        pooled = self.norm(pooled.squeeze(1))
        logits = self.classifier(self.dropout(pooled))
        out = {"logits": logits}
        if labels is not None:
            out["loss"] = nn.functional.cross_entropy(logits, labels)
        return out

model = WhisperAttnSER(BACKBONE, num_labels=len(LABELS), n_heads=N_HEADS)
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "f1_macro": f1_score(p.label_ids, preds, average="macro")}

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

args = TrainingArguments(
    output_dir="./whisper-ta-emotion",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    remove_unused_columns=False,   # keep attention_mask for our custom forward
    report_to=["none"],
    push_to_hub=bool(HF_TOKEN),
    hub_model_id=HUB_REPO,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=tamil_test,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# ===== speaker-independent Tamil-test report =====
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(tamil_test)
y, yhat = pred.label_ids, pred.predictions.argmax(1)
print(classification_report(y, yhat, target_names=LABELS, digits=3))
print("confusion matrix (rows=true):\n", confusion_matrix(y, yhat))

In [ ]:
# ===== save weights + push (custom nn.Module: state_dict + label map) =====
import json
os.makedirs("./whisper-ta-emotion", exist_ok=True)
torch.save(model.state_dict(), "./whisper-ta-emotion/pytorch_model.bin")
json.dump({"backbone": BACKBONE, "labels": LABELS, "n_heads": N_HEADS, "arch": "WhisperAttnSER"},
          open("./whisper-ta-emotion/ser_config.json", "w"))
fe.save_pretrained("./whisper-ta-emotion")
try:
    trainer.push_to_hub()
    print("Pushed:", HUB_REPO)
except Exception as e:
    print("push skipped:", e)

In [ ]:
# ===== quick inference demo =====
import librosa
@torch.no_grad()
def predict_emotion(path):
    y, _ = librosa.load(path, sr=16000, mono=True)
    out = fe(y, sampling_rate=16000, return_attention_mask=True, return_tensors="pt")
    dev = model.classifier.weight.device
    o = model(out.input_features.to(dev), attention_mask=out.attention_mask.to(dev))
    probs = o["logits"].softmax(-1)[0]
    i = int(probs.argmax())
    return id2label[i], float(probs[i])

# example: predict_emotion('/kaggle/input/<your-emota>/01_01_ang.wav')